# 🩺 USMLE & MedMCQA Subject Classifier Experimentation
This notebook:
1. Connects to `medmcqa.db` and `medqa_usmle.db`.
2. Inspects tables, schemas, and sample rows.
3. Tests zero-shot classification on labeled MedMCQA questions to evaluate accuracy on CPU without training.

### Imports and Folders

In [ ]:
import shutil
import time
from tqdm import tqdm
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Paths relative to the project root (or lab/notebooks/)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if "lab" in str(NOTEBOOK_DIR) else NOTEBOOK_DIR

MEDMCQA_DB = PROJECT_ROOT / "src" / "backend" / "datasets" / "medmcqa.db"
USMLE_DB = PROJECT_ROOT / "src" / "backend" / "datasets" / "medqa_usmle.db"

print(f"Project root: {PROJECT_ROOT}")
print(f"MedMCQA DB exists: {MEDMCQA_DB.exists()} ({MEDMCQA_DB})")
print(f"USMLE DB exists:   {USMLE_DB.exists()} ({USMLE_DB})")

### 1. Inspect Database Schemas & Tables
Let's see all tables, column names, data types, and row counts in both databases.

In [ ]:
def inspect_database(db_path: Path, name: str):
    print(f"\n{'='*20} {name} {'='*20}")
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        
        # Get all tables
        tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
        print(f"Tables: {[t[0] for t in tables]}")
        
        for table_name in [t[0] for t in tables]:
            print(f"\n--- Table: {table_name} ---")
            
            # Row count
            count = cursor.execute(f"SELECT count(*) FROM {table_name};").fetchone()[0]
            print(f"Total Rows: {count:,}")
            
            # Columns and Types
            columns = cursor.execute(f"PRAGMA table_info({table_name});").fetchall()
            col_df = pd.DataFrame(columns, columns=["cid", "name", "type", "notnull", "dflt_value", "pk"])
            display(col_df[["name", "type", "notnull", "pk"]])

inspect_database(MEDMCQA_DB, "MedMCQA (Labeled Indian Medical Exam Bank)")
inspect_database(USMLE_DB, "MedQA USMLE (Unlabeled Subjects)")

### 2. Sample Data Exploration
Let's pull sample rows from both to see the actual format of questions, options, explanations, and existing subjects.

In [ ]:
# Sample 3 rows from MedMCQA
with sqlite3.connect(MEDMCQA_DB) as conn:
    medmcqa_sample = pd.read_sql_query("SELECT * FROM bank_questions LIMIT 3;", conn)
print("MedMCQA Sample:")
display(medmcqa_sample)

# Sample 3 rows from USMLE
with sqlite3.connect(USMLE_DB) as conn:
    usmle_sample = pd.read_sql_query("SELECT * FROM bank_questions LIMIT 3;", conn)
display(usmle_sample)

In [ ]:
# Let's see the most common subjects in MedMCQA. These represent the labels we can test against.

with sqlite3.connect(MEDMCQA_DB) as conn:
    # 1. Subject counts
    subject_counts = pd.read_sql_query(
        "SELECT subject, COUNT(*) as count FROM bank_questions GROUP BY subject ORDER BY count DESC;", 
        conn
    )
    
    # 2. Topic counts (sorted by count descending)
    topic_counts = pd.read_sql_query(
        "SELECT topic, COUNT(*) as count FROM bank_questions GROUP BY topic ORDER BY count DESC;", 
        conn
    )

# Render side-by-side: Subjects on the Left, Topics on the Right
html_content = f"""
<div style="display: flex; gap: 30px; align-items: flex-start;">
    <div>
        <h4 style="margin-bottom: 8px;">Subjects ({len(subject_counts)})</h4>
        <div style="max-height: 500px; overflow-y: auto; border: 1px solid #444; border-radius: 6px; padding: 4px;">
            {subject_counts.to_html(index=False, classes='table')}
        </div>
    </div>
    <div>
        <h4 style="margin-bottom: 8px;">Topics ({len(topic_counts):,})</h4>
        <div style="max-height: 500px; overflow-y: auto; border: 1px solid #444; border-radius: 6px; padding: 4px;">
            {topic_counts.to_html(index=False, classes='table')}
        </div>
    </div>
</div>
"""

display(HTML(html_content))

### 3. Support Functions

In [ ]:
# Display function
def display_results(y_train, y_train_pred, y_test, y_test_pred):
    train_rep = classification_report(y_train, y_train_pred, output_dict=True, zero_division=0)
    test_rep = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    
    subjects = [s for s in test_rep.keys() if s not in ['accuracy', 'macro avg', 'weighted avg']]
    
    comparison_data = []
    for s in subjects:
        tr_f1 = train_rep[s]['f1-score'] * 100
        te_f1 = test_rep[s]['f1-score'] * 100
        comparison_data.append({
            'Subject': s,
            'Test Support': int(test_rep[s]['support']),
            'Train F1 (%)': round(tr_f1, 1),
            'Test F1 (%)': round(te_f1, 1),
            'Gap': round(tr_f1 - te_f1, 1),
            'Test Recall (%)': round(test_rep[s]['recall'] * 100, 1),
            'Test Precision (%)': round(test_rep[s]['precision'] * 100, 1)
        })
        
    comp_df = pd.DataFrame(comparison_data).sort_values(by='Test F1 (%)', ascending=False)
    
    # Clean styling: highlight top performers
    display(comp_df.style.background_gradient(subset=['Test F1 (%)'], cmap='Blues')
                         .background_gradient(subset=['Gap'], cmap='Reds')
                         .format(precision=1))


def classify_and_analyze(
    pipeline, 
    exam_filter: str, 
    min_margin: float = 0.10,
    fallback_label: str = "Multisystem & General Principles",
    db_path=USMLE_DB
):
    """
    Classifies question bank with a margin threshold fallback.
    Questions below the threshold are assigned a professional integrative label.
    """
    print(f"\n{'='*25} Classifying {exam_filter} {'='*25}")
    print(f"⚙️  Thresholds: min_margin >= {min_margin:.2f}")
    print(f"🏷️  Fallback Label: '{fallback_label}'")
    
    # 1. Fetch questions
    with sqlite3.connect(db_path) as conn:
        df_usmle = pd.read_sql_query(
            """
            SELECT id, question, opa, opb, opc, opd, exam_type 
            FROM bank_questions 
            WHERE exam_type = ?;
            """,
            conn,
            params=[exam_filter]
        )
        
    total_q = len(df_usmle)
    df_usmle['text'] = df_usmle.apply(format_text, axis=1)
    
    # 2. Decision function & probabilities
    t0 = time.time()
    decision_scores = pipeline.decision_function(df_usmle['text'])
    classes = pipeline.classes_
    probs = softmax(decision_scores, axis=1)
    
    # Top-1 and Runner-up calculation
    sorted_scores = np.sort(decision_scores, axis=1)
    top_margin = sorted_scores[:, -1]
    second_margin = sorted_scores[:, -2]
    margin_gap = top_margin - second_margin
    
    top_indices = np.argmax(decision_scores, axis=1)
    raw_predicted_subjects = classes[top_indices]
    top_confidences = probs[np.arange(total_q), top_indices]
    
    # 3. Apply the Fallback Rule
    # Question must pass BOTH confidence and margin gap to keep specific label
    passes_threshold = (top_margin >= min_margin)
    
    final_subjects = np.where(passes_threshold, raw_predicted_subjects, fallback_label)
    
    pred_time = time.time() - t0
    fallback_count = np.sum(~passes_threshold)
    print(f"✅ Classified {total_q:,} questions in {pred_time:.2f}s")
    print(f"🛡️  Assigned to '{fallback_label}': {fallback_count:,} ({fallback_count/total_q*100:.1f}%)\n")
    
    # Attach to DataFrame
    df_usmle['predicted_subject'] = final_subjects
    df_usmle['raw_subject'] = raw_predicted_subjects
    df_usmle['confidence'] = top_confidences
    df_usmle['margin'] = top_margin
    df_usmle['margin_gap'] = margin_gap
    df_usmle['is_fallback'] = ~passes_threshold
    
    # 4. Summary Table across all final categories (including fallback)
    all_categories = list(classes) + [fallback_label]
    stats = []
    
    for subj in set(final_subjects):
        sub_df = df_usmle[df_usmle['predicted_subject'] == subj]
        cnt = len(sub_df)
        pct = (cnt / total_q) * 100
        
        stats.append({
            'Subject': subj,
            'Count': cnt,
            'Share (%)': round(pct, 1),
            'Avg Conf (%)': round(sub_df['confidence'].mean() * 100, 1),
            'Std Conf (%)': round(sub_df['confidence'].std() * 100, 1),
            'Avg Margin': round(sub_df['margin'].mean(), 2),
            'Std Margin': round(sub_df['margin'].std(), 2),
            'Avg Gap': round(sub_df['margin_gap'].mean(), 2),
            'Std Gap': round(sub_df['margin_gap'].std(), 2),
        })
        
    summary_df = pd.DataFrame(stats).sort_values(by='Count', ascending=False)
    
    styled_table = (
        summary_df.style
        .background_gradient(subset=['Count'], cmap='Blues')
        .background_gradient(subset=['Avg Conf (%)'], cmap='Greens')
        .background_gradient(subset=['Avg Gap'], cmap='YlGnBu')
        .format({
            'Count': '{:,}',
            'Share (%)': '{:.1f}%',
            'Avg Conf (%)': '{:.1f}%',
            'Std Conf (%)': '{:.1f}%',
            'Avg Margin': '{:.2f}',
            'Std Margin': '{:.2f}',
            'Avg Gap': '{:.2f}',
            'Std Gap': '{:.2f}'
        })
    )
    
    display(styled_table)
    return df_usmle

### 4. USMLE Step 1 Classifier Training and Testing

In [ ]:

# ==========================================
# 1. STEP 1 TAXONOMY MAPPING (Easily Editable)
# ==========================================
# Map MedMCQA subject -> Target USMLE Step 1 Discipline
STEP1_MAPPING = {
    'Pathology': 'Pathology',
    'Physiology': 'Physiology',
    'Anatomy': 'Gross Anatomy & Embryology',
    'Microbiology': 'Microbiology',
    'Pharmacology': 'Pharmacology',
    'Social & Preventive Medicine': 'Behavioral Sciences',
    'Biochemistry': 'Biochemistry',
}

# 2. Load Step 1 eligible questions from MedMCQA
print("Loading Step 1 training data from MedMCQA...")
placeholders = ",".join(["?"] * len(STEP1_MAPPING))
with sqlite3.connect(MEDMCQA_DB) as conn:
    df_step1 = pd.read_sql_query(
        f"""
        SELECT question, opa, opb, opc, opd, subject 
        FROM bank_questions 
        WHERE subject IN ({placeholders})
        ORDER BY RANDOM()
        LIMIT 45000;
        """,
        conn,
        params=list(STEP1_MAPPING.keys())
    )

# Apply mapping and format input text
df_step1['target_subject'] = df_step1['subject'].map(STEP1_MAPPING)

def format_text(row):
    q = str(row['question'] or '')
    a = str(row['opa'] or '')
    b = str(row['opb'] or '')
    c = str(row['opc'] or '')
    d = str(row['opd'] or '')
    return f"{q} A: {a} B: {b} C: {c} D: {d}"

df_step1['text'] = df_step1.apply(format_text, axis=1)

print(f"Loaded {len(df_step1):,} questions across {df_step1['target_subject'].nunique()} Step 1 Disciplines.")
print(df_step1['target_subject'].value_counts())

# 3. Train / Test Split
X_tr_s1, X_te_s1, y_tr_s1, y_te_s1 = train_test_split(
    df_step1['text'], df_step1['target_subject'], test_size=0.2, random_state=42, stratify=df_step1['target_subject']
)

# 4. Pipeline for Step 1
svc_step1 = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=55000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.75,
        stop_words='english',
        sublinear_tf=True
    )),
    ('clf', LinearSVC(dual='auto', C=0.8, max_iter=2500, random_state=42))
])

# 5. Train & Evaluate Step 1
print("\nTraining Step 1 Classifier on CPU...")
t0 = time.time()
svc_step1.fit(X_tr_s1, y_tr_s1)
tr_time = time.time() - t0

y_tr_pred = svc_step1.predict(X_tr_s1)
y_te_pred = svc_step1.predict(X_te_s1)

tr_acc = accuracy_score(y_tr_s1, y_tr_pred) * 100
te_acc = accuracy_score(y_te_s1, y_te_pred) * 100

print(f"✅ Training Time: {tr_time:.2f}s")
print(f"🎯 Step 1 Train Acc: {tr_acc:.2f}% | Test Acc: {te_acc:.2f}% | Gap: {tr_acc - te_acc:+.2f}%\n")

# Evaluation breakdown
display_results(y_tr_s1, y_tr_pred, y_te_s1, y_te_pred)

### 5. Classify USMLE Step 1 Questions

In [ ]:
# Run classifier on all 7,009 USMLE Step 1 questions
usmle_step1_classified = classify_and_analyze(
    pipeline=svc_step1, 
    exam_filter='USMLE Step 1',
    min_margin=0,
)

### 6. USMLE Step 2&3 Classifier Training and Testing

In [ ]:
# ==========================================
# 1. STEP 2 & 3 TAXONOMY MAPPING (Official USMLE)
# ==========================================
STEP2_3_MAPPING = {
    'Medicine': 'Medicine',
    'Skin': 'Medicine',
    'Radiology': 'Medicine',
    'Surgery': 'Surgery',
    'Orthopaedics': 'Surgery',
    'ENT': 'Surgery',
    'Ophthalmology': 'Surgery',
    'Anaesthesia': 'Surgery',
    'Pediatrics': 'Pediatrics',
    'Gynaecology & Obstetrics': 'Obstetrics & Gynecology',
    'Psychiatry': 'Psychiatry'
}

# 2. Load Step 2/3 eligible questions from MedMCQA
print("Loading Step 2 & 3 training data from MedMCQA...")
placeholders_s2 = ",".join(["?"] * len(STEP2_3_MAPPING))
with sqlite3.connect(MEDMCQA_DB) as conn:
    df_step2 = pd.read_sql_query(
        f"""
        SELECT question, opa, opb, opc, opd, subject 
        FROM bank_questions 
        WHERE subject IN ({placeholders_s2})
        ORDER BY RANDOM()
        LIMIT 45000;
        """,
        conn,
        params=list(STEP2_3_MAPPING.keys())
    )

df_step2['target_subject'] = df_step2['subject'].map(STEP2_3_MAPPING)
df_step2['text'] = df_step2.apply(format_text, axis=1)

print(f"Loaded {len(df_step2):,} questions across {df_step2['target_subject'].nunique()} Clinical Sciences.")
print(df_step2['target_subject'].value_counts())

# 3. Train / Test Split
X_tr_s2, X_te_s2, y_tr_s2, y_te_s2 = train_test_split(
    df_step2['text'], df_step2['target_subject'], test_size=0.2, random_state=42, stratify=df_step2['target_subject']
)

# 4. Pipeline for Step 2 & 3
svc_step2 = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=55000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.75,
        stop_words='english',
        sublinear_tf=True
    )),
    ('clf', LinearSVC(dual='auto', C=0.8, max_iter=2500, random_state=42))
])

# 5. Train & Evaluate on Test Set
print("\nTraining Step 2 & 3 Classifier on CPU...")
t0 = time.time()
svc_step2.fit(X_tr_s2, y_tr_s2)
tr_time_s2 = time.time() - t0

y_tr_pred_s2 = svc_step2.predict(X_tr_s2)
y_te_pred_s2 = svc_step2.predict(X_te_s2)

tr_acc_s2 = accuracy_score(y_tr_s2, y_tr_pred_s2) * 100
te_acc_s2 = accuracy_score(y_te_s2, y_te_pred_s2) * 100

print(f"✅ Training Time: {tr_time_s2:.2f}s")
print(f"🎯 Step 2 & 3 Train Acc: {tr_acc_s2:.2f}% | Test Acc: {te_acc_s2:.2f}% | Gap: {tr_acc_s2 - te_acc_s2:+.2f}%\n")

# Display Train vs Test report using your display function
display_results(y_tr_s2, y_tr_pred_s2, y_te_s2, y_te_pred_s2)

### 7. Classify USMLE Step 2 & 3 Questions

In [ ]:
# Run on all 5,714 USMLE Step 2 and Step 3 questions
usmle_step2_classified = classify_and_analyze(
    pipeline=svc_step2,
    exam_filter='USMLE Step 2 and Step 3',
    min_margin=0.0,
    fallback_label="Multisystem & Ambulatory Care"
)

### 8. Update USMLE Database Entries with Subjects

In [42]:
# 1. Create a safe backup before modifying
backup_path = USMLE_DB.with_suffix(".db.bak")
if not backup_path.exists():
    shutil.copy2(USMLE_DB, backup_path)
    print(f"📦 Backup created at: {backup_path}")
else:
    print(f"ℹ️ Backup already exists at: {backup_path}")

# 2. Combine predictions from both models
# Each dataframe has 'id' and 'predicted_subject'
update_records = []

# Step 1
for _, row in usmle_step1_classified.iterrows():
    update_records.append((row['predicted_subject'], str(row['id'])))

# Step 2 & 3
for _, row in usmle_step2_classified.iterrows():
    update_records.append((row['predicted_subject'], str(row['id'])))

print(f"Total records prepared for database update: {len(update_records):,}")

# 3. Batch update the database
with sqlite3.connect(USMLE_DB) as conn:
    cursor = conn.cursor()
    cursor.executemany(
        "UPDATE bank_questions SET subject = ? WHERE id = ?;",
        update_records
    )
    conn.commit()
    print("✅ Database successfully updated!")

# 4. Verification Check
with sqlite3.connect(USMLE_DB) as conn:
    verification_df = pd.read_sql_query(
        """
        SELECT 
            exam_type, 
            subject, 
            COUNT(*) as question_count 
        FROM bank_questions 
        GROUP BY exam_type, subject 
        ORDER BY exam_type, question_count DESC;
        """,
        conn
    )
    
    null_count = conn.execute("SELECT count(*) FROM bank_questions WHERE subject IS NULL OR subject = '';").fetchone()[0]

print(f"\nRemaining unclassified / NULL questions: {null_count}")
display(verification_df)

📦 Backup created at: /home/semere/sources/usmle-study-helper/src/backend/datasets/medqa_usmle.db.bak
Total records prepared for database update: 12,723
✅ Database successfully updated!

Remaining unclassified / NULL questions: 0


,exam_type,subject,question_count
0,USMLE Step 1,Pathology,2463
1,USMLE Step 1,Multisystem & General Principles,2317
2,USMLE Step 1,Pharmacology,565
3,USMLE Step 1,Microbiology,409
4,USMLE Step 1,Physiology,407
5,USMLE Step 1,Gross Anatomy & Embryology,371
6,USMLE Step 1,Behavioral Sciences,248
7,USMLE Step 1,Biochemistry,229
8,USMLE Step 2 and Step 3,Medicine,2552
9,USMLE Step 2 and Step 3,Multisystem & Ambulatory Care,1141
